By the end of this notebook, below questions will be answered:

1. What is an embedding?
2. How do we generate embeddings?
3. Where are embeddings stored?
4. Why do we need embeddings before Vector Search?
5. Why is the same embedding model used for documents and questions?


In [0]:

Customer Notes
       ↓
Embedding Model
       ↓
Vectors
       ↓
Delta Table

In [0]:
#Step 1: Create Sample Customer Notes
customer_notes = [
    (1001, "Customer wants to cancel service"),
    (1002, "Customer requesting plan upgrade"),
    (1003, "Internet connection is unstable"),
    (1004, "Customer has billing complaint"),
    (1005, "Customer wants to terminate account"),
    (1006, "Slow internet speed reported"),
    (1007, "Interested in premium package"),
    (1008, "Customer unhappy with service quality")
]

notes_df = spark.createDataFrame(customer_notes, ["customer_id", "note"])
display(notes_df)

#Step 2: Save to Unity Catalog
notes_df.write.format("delta").mode("overwrite").saveAsTable("dbw_agentic_ai_dev.telco_ai.customer_notes")
#display(spark.table("dbw_agentic_ai_dev.telco_ai.customer_notes"))

#Step 3: Choose Embedding Model and Generate Embeddings

from databricks.sdk import WorkspaceClient
from databricks.sdk.service.serving import ChatMessage

w = WorkspaceClient()

#now load notes
notes = (spark.table("dbw_agentic_ai_dev.telco_ai.customer_notes").toPandas())

#generate embeddings
embedding_rows = []

for row in notes.itertuples():
    response = w.serving_endpoints.query(name="databricks-gte-large-en", input=[row.note])
    embedding = response.data[0].embedding
    embedding_rows.append((row.customer_id, row.note, [float(x) for x in embedding]))

# Step 4: Create Embedding Table
from pyspark.sql.types import StructType, StructField, IntegerType, StringType, ArrayType, DoubleType

schema = StructType([
    StructField("customer_id", IntegerType(), True),
    StructField("note", StringType(), True),
    StructField("embedding", ArrayType(DoubleType()), True)
])

embedding_df = spark.createDataFrame(
    embedding_rows,
    schema=schema
)

#display(embedding_df)

#Step 5: Save Embeddings
embedding_df.write.format("delta").mode("overwrite").saveAsTable("dbw_agentic_ai_dev.telco_ai.customer_note_embeddings")

# Step 6: Verify
display(spark.table("dbw_agentic_ai_dev.telco_ai.customer_note_embeddings"))

By the end of this notebook, below questions will be answered:

1. What is an embedding? --> Numerical representation of text meaning.
2. How do we generate embeddings? --> Pass text to an embedding model and receive a vector representing the meaning of that text.
3. Where are embeddings stored? -->Delta table.
4. Why do we need embeddings before Vector Search? -->Vector Search works on vectors, not text.
5. Why is the same embedding model used for documents and questions? --->So both live in the same vector space and similarity search works.
